# FHOPS Onboarding — Stochastic What-If

This notebook demonstrates stochastic playback with sampling to assess
schedule robustness under downtime and weather uncertainty.

## Stochastic playback

1. **Base deterministic playback** — the same pipeline as notebook 02.
2. **SamplingConfig** — configure downtime probability, weather probability,
   landing shocks, and sample count.
3. **`run_stochastic_playback(pb, assignments, sampling_config)`** — runs
   the base deterministic playback, then overlays stochastic events on
   each sample, producing an `EnsembleResult`.
4. **Aggregate** — `shift_dataframe_from_ensemble()` and `day_dataframe_from_ensemble()`
   concatenate results across all samples (and optionally the base).

## Event models

| Event | Parameters |
| --- | --- |
| Downtime | `probability` per eligible assignment, `max_concurrent` per day |
| Weather | `probability` per day, `severity` (production fraction), `window` (consecutive days) |
| Landing shock | `probability` per day, `multiplier_range`, `duration` (consecutive days) |

## Budget

- Scenario: `med42` (42-day horizon, 12 blocks) — large enough to show
  meaningful variability but tractable.
- Solve: 200 SA iterations, seed 7.
- Samples: 10 stochastic realisations with modest downtime/weather probabilities.

No licensed solver is required.

In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd() / "examples"))
from notebook_support import find_repo_root, discover_scenarios, run_fhops_cli
from fhops.scenario.io import load_scenario
from fhops.scenario.contract import Problem
from fhops.optimization.heuristics import solve_sa
from fhops.evaluation import (
    run_playback,
    run_stochastic_playback,
    SamplingConfig,
    shift_dataframe_from_ensemble,
    day_dataframe_from_ensemble,
)
from fhops.evaluation.playback.events import (
    DowntimeEventConfig,
    WeatherEventConfig,
    LandingShockConfig,
)
import pandas as pd

repo_root = find_repo_root()
print(f'Repo root: {repo_root}')

Repo root: /srv/shared-data/gep/jupyterhub04-projects/fhops


In [2]:
# Load med42 scenario
med42_path = Path(repo_root) / 'examples' / 'med42' / 'scenario.yaml'
sc = load_scenario(str(med42_path))
pb = Problem.from_scenario(sc)
print(f'Scenario: {sc.name}')
print(f'  Days: {sc.num_days}, Blocks: {len(sc.blocks)}, '
      f'Machines: {len(sc.machines)}, Landings: {len(sc.landings)}')
print(f'  Harvest systems: {list(sc.harvest_systems.keys())[:3]}')
print(f'  Shifts: {sorted({s.shift_id for s in pb.shifts})}')

Scenario: FHOPS Medium42
  Days: 42, Blocks: 12, Machines: 9, Landings: 12
  Harvest systems: ['ground_fb_skid']
  Shifts: ['S1']


In [3]:
# Solve med42 with SA
print('Solving med42 with 200 SA iterations...', flush=True)
sa_result = solve_sa(pb, iters=200, seed=7)
assignments = sa_result['assignments']
print(f'  objective={sa_result["objective"]:.3f}  rows={len(assignments)}')
display(assignments[['machine_id', 'block_id', 'day', 'shift_id']].head(10))

Solving med42 with 200 SA iterations...
  objective=-37435.427  rows=211


,machine_id,block_id,day,shift_id
0,H1,B04,1,S1
21,H2,B12,1,S1
1,H1,B12,2,S1
22,H2,B09,2,S1
42,H3,B04,2,S1
2,H1,B10,3,S1
23,H2,B12,3,S1
43,H3,B09,3,S1
83,H4,B04,3,S1
123,H5,B04,3,S1


## Deterministic baseline

Run a single deterministic playback as the baseline for comparison.

In [4]:
from fhops.evaluation import compute_kpis

base_result = run_playback(pb, assignments)
base_kpis = compute_kpis(pb, assignments)
print(f'Baseline KPIs:')
print(f'  total_production      : {base_kpis["total_production"]:.2f}')
print(f'  completed_blocks      : {base_kpis["completed_blocks"]}')
print(f'  mobilisation_cost     : {base_kpis["mobilisation_cost"]:.2f}')
print(f'  sequencing_violations : {base_kpis["sequencing_violation_count"]}')

Baseline KPIs:
  total_production      : 31979.75
  completed_blocks      : 8.0
  mobilisation_cost     : 10181.20
  sequencing_violations : 0


## Stochastic sampling config

Set modest event probabilities to model typical operational uncertainty:

In [5]:
sampling_config = SamplingConfig(
    samples=10,
    base_seed=42,
    downtime=DowntimeEventConfig(
        probability=0.08,       # 8% chance of downtime per eligible assignment
        max_concurrent=None,    # binomial sampling
    ),
    weather=WeatherEventConfig(
        day_probability=0.05,   # 5% chance of weather per day
        severity_levels={"moderate": 0.30},  # 30% production reduction
        impact_window_days=1,    # 1 consecutive day affected
    ),
    landing=LandingShockConfig(
        probability=0.03,       # 3% chance of landing shock per day
        capacity_multiplier_range=(0.4, 0.8),  # min/max throughput multiplier
        duration_days=1,          # 1 consecutive day
    ),
)
print(f'SamplingConfig: {sampling_config.samples} samples, '
      f'downtime_prob={sampling_config.downtime.probability}, '
      f'weather_prob={sampling_config.weather.day_probability}')

SamplingConfig: 10 samples, downtime_prob=0.08, weather_prob=0.05


## Run stochastic playback

Each sample applies random downtime/weather/landing events to the base schedule.

In [6]:
ensemble = run_stochastic_playback(pb, assignments, sampling_config=sampling_config)
print(f'Ensemble: {len(ensemble.samples)} samples + base result')
print(f'Base production: {ensemble.base_result.delivered_total:.2f}')

Ensemble: 10 samples + base result
Base production: 31979.75


## Day-level ensemble summary

Concatenate day summaries across all samples to see production variability.

In [7]:
day_df = day_dataframe_from_ensemble(ensemble, include_base=True)
display(day_df.head(20))

,day,sample_id,production_units,total_hours,idle_hours,mobilisation_cost,completed_blocks,blackout_conflicts,sequencing_violations,available_hours,utilisation_ratio,downtime_hours,downtime_events,weather_severity_total
0,1,0,2476.322873,48.0,168.0,0.00,0,0,0,216.0,0.222222,0.0,0,0.0
1,2,0,3628.256368,72.0,144.0,108.64,0,0,0,216.0,0.333333,0.0,0,0.0
2,3,0,4000.710821,120.0,96.0,161.40,0,0,0,216.0,0.555556,0.0,0,0.0
3,4,0,4227.068211,120.0,96.0,160.68,0,0,0,216.0,0.555556,0.0,0,0.0
4,5,0,2466.833430,96.0,120.0,212.96,0,0,0,216.0,0.444444,0.0,0,0.0
5,6,0,2729.301941,96.0,120.0,214.16,0,0,0,216.0,0.444444,0.0,0,0.0
6,7,0,4075.915216,120.0,96.0,216.32,0,0,0,216.0,0.555556,0.0,0,0.0
7,8,0,4093.333529,120.0,96.0,161.40,0,0,0,216.0,0.555556,0.0,0,0.0
8,9,0,4438.363325,144.0,72.0,161.64,0,0,0,216.0,0.666667,0.0,0,0.0
9,10,0,5309.405049,168.0,48.0,270.28,0,0,0,216.0,0.777778,0.0,0,0.0


## Summary statistics across samples

Aggregate production, mobilisation cost, and sequencing violations
across all stochastic samples.

In [8]:
sample_stats = day_df.groupby('sample_id').agg(
    total_production=('production_units', 'sum'),
    total_mobilisation_cost=('mobilisation_cost', 'sum'),
    total_sequencing_violations=('sequencing_violations', 'sum'),
    mean_utilisation=('utilisation_ratio', 'mean'),
).reset_index()

summary = pd.DataFrame([
    {'metric': 'total_production', 'mean': sample_stats['total_production'].mean(),
     'std': sample_stats['total_production'].std(), 'min': sample_stats['total_production'].min(),
     'max': sample_stats['total_production'].max()},
    {'metric': 'total_mobilisation_cost', 'mean': sample_stats['total_mobilisation_cost'].mean(),
     'std': sample_stats['total_mobilisation_cost'].std(), 'min': sample_stats['total_mobilisation_cost'].min(),
     'max': sample_stats['total_mobilisation_cost'].max()},
    {'metric': 'sequencing_violations', 'mean': sample_stats['total_sequencing_violations'].mean(),
     'std': sample_stats['total_sequencing_violations'].std(), 'min': int(sample_stats['total_sequencing_violations'].min()),
     'max': int(sample_stats['total_sequencing_violations'].max())},
    {'metric': 'mean_utilisation', 'mean': sample_stats['mean_utilisation'].mean(),
     'std': sample_stats['mean_utilisation'].std(), 'min': sample_stats['mean_utilisation'].min(),
     'max': sample_stats['mean_utilisation'].max()},
])
summary.round(3)

,metric,mean,std,min,max
0,total_production,131045.618,46674.911,108365.894,263079.661
1,total_mobilisation_cost,10410.208,3255.697,9160.760,19659.960
2,sequencing_violations,31.900,6.064,19.000,40.000
3,mean_utilisation,0.517,0.012,0.505,0.542


## Comparison: baseline vs stochastic mean

The stochastic ensemble shows how production and mobilisation vary
under random downtime and weather events.

In [9]:
comparison = pd.DataFrame([
    {'scenario': 'deterministic', 'total_production': base_kpis['total_production'],
     'mobilisation_cost': base_kpis['mobilisation_cost'],
     'sequencing_violations': base_kpis['sequencing_violation_count']},
    {'scenario': 'stochastic (mean)', 'total_production': sample_stats['total_production'].mean(),
     'mobilisation_cost': sample_stats['total_mobilisation_cost'].mean(),
     'sequencing_violations': sample_stats['total_sequencing_violations'].mean()},
    {'scenario': 'stochastic (min)', 'total_production': sample_stats['total_production'].min(),
     'mobilisation_cost': sample_stats['total_mobilisation_cost'].min(),
     'sequencing_violations': int(sample_stats['total_sequencing_violations'].min())},
    {'scenario': 'stochastic (max)', 'total_production': sample_stats['total_production'].max(),
     'mobilisation_cost': sample_stats['total_mobilisation_cost'].max(),
     'sequencing_violations': int(sample_stats['total_sequencing_violations'].max())},
]).round(2)
comparison

,scenario,total_production,mobilisation_cost,sequencing_violations
0,deterministic,31979.75,10181.20,0.0
1,stochastic (mean),131045.62,10410.21,31.9
2,stochastic (min),108365.89,9160.76,19.0
3,stochastic (max),263079.66,19659.96,40.0


## Per-sample production distribution

A compact table showing each sample's total production.

In [10]:
sample_production = sample_stats[['sample_id', 'total_production']].copy()
sample_production['deviation_from_baseline'] = (
    sample_production['total_production'] - base_kpis['total_production']
)
sample_production['deviation_pct'] = (
    sample_production['deviation_from_baseline'] / base_kpis['total_production'] * 100
)
sample_production.sort_values('sample_id').round(2)

,sample_id,total_production,deviation_from_baseline,deviation_pct
0,0,263079.66,231099.91,722.64
1,1,116538.91,84559.16,264.41
2,2,123454.05,91474.30,286.04
3,3,125128.04,93148.28,291.27
4,4,113577.66,81597.90,255.15
5,5,116377.96,84398.21,263.91
6,6,116502.84,84523.09,264.30
7,7,117183.37,85203.62,266.43
8,8,108365.89,76386.14,238.86
9,9,110247.80,78268.05,244.74


## Summary

- Stochastic playback overlays random downtime, weather, and landing events
  on a base deterministic schedule.
- The ensemble reveals production variability and sequencing vulnerability.
- `SamplingConfig` parameters control event probability, severity, and duration.
- This series is complete: orientation -> solve/compare -> playback/KPIs -> stochastic what-if.

## Complete series

1. `00_fhops_orientation.ipynb` — locate repo, load scenarios, CLI validate, AAM helpers.
2. `01_fhops_operations_simulation.ipynb` — inspect and replay the operational simulation core.
3. `02_fhops_solve_compare.ipynb` — solve tiny7/small21 via API + CLI, compare objectives/KPIs.
4. `03_fhops_playback_kpis.ipynb` — run playback, shift/day DataFrames, compute KPIs.
5. `04_fhops_stochastic_what_if.ipynb` — med42 stochastic ensemble with sampling.